### Reproduction of Practical work paper with a different(newer) dataset

Possible dataset candidates:
- Yelp open dataset(info about restaurants, hotels etc, geospatial data in general)
- Goodreads dataset(sparse dataset that was used in the paper)
- Amazon reviews 2023 dataset

In [3]:

import pandas as pd

import scipy

Installing elliot

In [4]:
# !git clone https://github.com/sisinflab/elliot.git
# !cd elliot
# !conda create --name elliot_env python=3.8 -y
# !conda activate elliot_env
# !pip install --upgrade pip
# !pip install -e .

## 1. Yelp open dataset

Text fields are irrelevant for now. We need 3 files: businnes, review and user json files

The logic is to pull out 500k reviews and based on that pull out users and businesses from other two datasets for interaction matrix to be more dense(less sparse)

In [5]:
#yelp_business = pd.read_json("data/yelp/yelp_academic_dataset_business.json",lines = True, nrows=500000)
yelp_reviews = pd.read_json("data/yelp/yelp_academic_dataset_review.json",lines = True, nrows=500000)
#yelp_user = pd.read_json("data/yelp/yelp_academic_dataset_user.json", lines = True, nrows=500000)

valid_users = set(yelp_reviews['user_id'])
valid_businesses = set(yelp_reviews['business_id'])

business_chunks = []
for chunk in pd.read_json("data/yelp/yelp_academic_dataset_business.json", lines=True, chunksize=100000):
    matched = chunk[chunk['business_id'].isin(valid_businesses)]
    business_chunks.append(matched)

yelp_business = pd.concat(business_chunks)

user_chunks = []
for chunk in pd.read_json("data/yelp/yelp_academic_dataset_user.json", lines=True, chunksize=100000):
    matched = chunk[chunk['user_id'].isin(valid_users)]
    user_chunks.append(matched)

yelp_user = pd.concat(user_chunks)





In [6]:
yelp_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [7]:
yelp_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [8]:
yelp_reviews= yelp_reviews.drop(labels=["text"],axis=1) # Text column is irrelevant for now

In [9]:
yelp_user.head()

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0


Loading Yelp dataset as an Elliot dataloader

Building an interaction matrix

In [10]:
print(yelp_reviews["user_id"].nunique())
print(yelp_reviews["business_id"].nunique())


318303
14768


300k unique users and 15k unique businesses, the interaction matrix hence is UxL while number of reviews we dug out is 500k, so on average 1 user interacts with 1.4 items out of 15k. Elliot considers datasets as .tsv files of interactions with columns: UserID, ItemID, Rating, [Timestamp]

In [11]:
yelp_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,2017-01-14 20:54:15


In [12]:
yelp_reviews= yelp_reviews.drop(labels=["review_id", "useful","funny","cool"],axis=1) 
yelp_reviews.head()


,user_id,business_id,stars,date
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,2018-07-07 22:09:11
1,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,2012-01-03 15:28:18
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,2014-02-05 20:30:30
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,2015-01-04 00:01:03
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,2017-01-14 20:54:15


In [13]:
yelp_reviews.to_csv("data/yelp/yelp.tsv", sep="\t")

Now we should write our own config file to reproduce experiments

Explanatory variables used in paper:



Algorithms used in paper:
- User-KNN cosine
- User-KNN Pearson
- User-KNN amplified
- User-KNN IDF
- User-KNN BM25
- Item-KNN Cosine
- Item-KNN Pearson
- Item-KNN adjusted


-------------------------
- MF
- SVD
- BPR-MF
- NMF
- PMF

<div>
experiment:
  version: 0.3.1
  dataset: yelp_reviews
  data_config:
    strategy: dataset
    dataset_path: ../data/yelp/yelp.tsv 
    side_information:
      - dataloader: ChainedKG
        map: ../data/yelp/yelp.tsv 
        features: ../data/yelp/yelp.tsv 
#  prefiltering:
#    strategy: user_average # Not applied in the paper experiments
  splitting:
    save_on_disk: True
    save_folder: ../data/yelp/splits
    test_splitting:
        strategy: temporal_hold_out
        test_ratio: 0.2
    validation_splitting:
        strategy: temporal_hold_out
        test_ratio: 0.2
  top_k: 50
  evaluation:
    cutoffs: [10, 5]
    simple_metrics: [nDCG,Precision,ItemCoverage,EPC,Gini]
    relevance_threshold: 1
  gpu: 1
  external_models_path: ../external/models/__init__.py
  models:
    Random:
      meta:
        verbose: True
        save_recs: True
      seed: 42
    external.MostPop:
      meta:
        verbose: True
        save_recs: True
        validation_metric: nDCG@10
    UserKNN:
      meta:
        save_recs: True
      neighbors: 40
      similarity: cosine
      implementation: aiolli
    ItemKNN:
      meta:
        verbose: True
        save_recs: True
        validation_metric: nDCG@10
      neighbors: [50, 70, 100]
      similarity: [cosine, euclidean]
      implementation: standard
    AttributeItemKNN:
      meta:
        verbose: True
        save_recs: True
        validation_metric: nDCG@10
      loader: ChainedKG
      neighbors: [50, 70, 100]
      similarity: [braycurtis, manhattan]
    MF:
      meta:
        save_recs: True
      epochs: 10
      batch_size: 512
      factors: 10
      lr: 0.001
      reg: 0.1
    PureSVD:
      meta:
        save_recs: True
      loader: ChainedKG
      factors: 10
      seed: 42
    BPRMF:
      meta:
        save_recs: True
      epochs: 10
      factors: 10
      lr: 0.001
      bias_regularization: 0
      user_regularization: 0.0025
      positive_item_regularization: 0.0025
      negative_item_regularization: 0.0025
      update_negative_item_factors: True
      update_users: True
      update_items: True
      update_bias: True
    NonNegMF:
      meta:
        save_recs: True
      epochs: 10
      batch_size: 512
      factors: 10
      lr: 0.001
      reg: 0.1
    PMF:
      meta:
        save_recs: True
      epochs: 10
      batch_size: 512
      factors: 50
      lr: 0.001
      reg: 0.0025
      gaussian_variance: 0.1

</div>



In [ ]:
"""Convert data/yelp/yelp.tsv into the formats Elliot expects.

Outputs (in data/yelp/):
  yelp_elliot.tsv      user  item  rating  timestamp   (ints, no header)
  item_attributes.tsv  item  cat1  cat2 ...            (business categories)
  user_ids.tsv, item_ids.tsv, category_ids.tsv         (int -> original id/name)
"""
import json
import os

import pandas as pd

DATA = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data", "yelp")

ratings = pd.read_csv(os.path.join(DATA, "yelp.tsv"), sep="\t", index_col=0)

users = {u: i for i, u in enumerate(ratings["user_id"].unique())}
items = {b: i for i, b in enumerate(ratings["business_id"].unique())}

out = pd.DataFrame({
    "user": ratings["user_id"].map(users),
    "item": ratings["business_id"].map(items),
    "rating": ratings["stars"],
    "timestamp": pd.to_datetime(ratings["date"]).astype("int64") // 10 ** 9,
})
out.to_csv(os.path.join(DATA, "yelp_elliot.tsv"), sep="\t", header=False, index=False)

categories = {}
attribute_lines = []
with open(os.path.join(DATA, "yelp_academic_dataset_business.json"), encoding="utf8") as f:
    for line in f:
        business = json.loads(line)
        if business["business_id"] not in items or not business["categories"]:
            continue
        cats = [c.strip() for c in business["categories"].split(",") if c.strip()]
        cat_ids = sorted({categories.setdefault(c, len(categories)) for c in cats})
        attribute_lines.append("\t".join(map(str, [items[business["business_id"]], *cat_ids])))

with open(os.path.join(DATA, "item_attributes.tsv"), "w", encoding="utf8") as f:
    f.write("\n".join(attribute_lines) + "\n")

pd.Series(list(users)).to_csv(os.path.join(DATA, "user_ids.tsv"), sep="\t", header=False)
pd.Series(list(items)).to_csv(os.path.join(DATA, "item_ids.tsv"), sep="\t", header=False)
pd.Series(list(categories)).to_csv(os.path.join(DATA, "category_ids.tsv"), sep="\t", header=False)

print(f"{len(out)} ratings, {len(users)} users, {len(items)} items, "
      f"{len(attribute_lines)} items with categories, {len(categories)} categories")


In [19]:
from elliot.run import run_experiment


run_experiment("yelp_configuration.yml")


2026-10-30 16:13:43.729637: I Start experiment
2026-10-30 16:13:43.732637: I There will be the splitting
2026-10-30 16:13:44.148755: I c:\Users\GAMINGPC.STORE\Desktop\JKU\scarcity-recsys\data\yelp\yelp_elliot.tsv - Loaded

Realized a Train/Validation Test splitting strategy

2026-10-30 16:13:54.709353: I Test Fold 0 - Validation Fold 0
2026-10-30 16:14:14.585524: I Statistics	Users:	31179	Items:	9866	Transactions:	83602	Sparsity:	0.9997282225784588
2026-10-30 16:14:41.596780: I Training begun for Random\n
2026-10-30 16:14:41.596780: I Loading parameters
2026-10-30 16:14:41.596780: I Parameter random_seed set to 42
2026-10-30 16:14:41.597785: I Hyperparameters:
2026-10-30 16:14:41.599785: I meta set to namespace(save_recs=True, verbose=True)
2026-10-30 16:14:41.601785: I seed set to 42
2026-10-30 16:14:41.604295: I Exploration: Test Fold exploration number 1
2026-10-30 16:14:41.606366: I Exploration: Train-Validation Fold exploration number 1
2026-10-30 16:14:50.510946: I 
2026-10-30 16

## Results: Yelp reproduction

Temporal hold-out split (20% test, 20% validation), relevance threshold 1. Hyperparameters are selected by nDCG@10 on validation. Gini is Elliot's version: higher means recommendations are spread more evenly across items.

### Best configuration per model, cutoff @10

| Model | Best config | nDCG | Precision | ItemCoverage | EPC | Gini |
|---|---|---:|---:|---:|---:|---:|
| Random | seed=42 | 0.0005 | 0.0001 | 9866 | 0.0001 | **0.8997** |
| MostPop | – | 0.0111 | 0.0031 | 15 | 0.0032 | 0.0009 |
| UserKNN | nn=40, cosine | 0.0308 | 0.0074 | 8166 | 0.0084 | 0.2473 |
| ItemKNN | nn=100, cosine | 0.0146 | 0.0042 | 8668 | 0.0043 | 0.3693 |
| AttributeItemKNN | nn=70, braycurtis | 0.0022 | 0.0006 | 9727 | 0.0007 | 0.4499 |
| MF | f=10 | 0.0080 | 0.0021 | 795 | 0.0023 | 0.0210 |
| PureSVD | f=10 | **0.0358** | **0.0090** | 173 | **0.0103** | 0.0053 |
| BPRMF | f=10 | 0.0130 | 0.0032 | 25 | 0.0035 | 0.0011 |
| NonNegMF | f=10 | 0.0016 | 0.0004 | 746 | 0.0005 | 0.0231 |
| PMF | f=50 | 0.0084 | 0.0023 | 1277 | 0.0025 | 0.0287 |

### Best configuration per model, cutoff @5

| Model | nDCG | Precision | ItemCoverage | EPC | Gini |
|---|---:|---:|---:|---:|---:|
| Random | 0.0003 | 0.0001 | 9866 | 0.0001 | **0.8591** |
| MostPop | 0.0078 | 0.0034 | 9 | 0.0035 | 0.0004 |
| UserKNN | 0.0236 | 0.0091 | 7124 | 0.0098 | 0.2152 |
| ItemKNN | 0.0097 | 0.0043 | 8183 | 0.0044 | 0.3262 |
| AttributeItemKNN | 0.0016 | 0.0007 | 9381 | 0.0007 | 0.4182 |
| MF | 0.0059 | 0.0024 | 536 | 0.0025 | 0.0139 |
| PureSVD | **0.0283** | **0.0115** | 99 | **0.0123** | 0.0029 |
| BPRMF | 0.0095 | 0.0037 | 17 | 0.0039 | 0.0005 |
| NonNegMF | 0.0011 | 0.0005 | 734 | 0.0005 | 0.0204 |
| PMF | 0.0060 | 0.0026 | 949 | 0.0028 | 0.0204 |

### Observations

- **PureSVD is the most accurate model** (nDCG@10 = 0.036), but it only recommends 173 distinct items, so most of its accuracy comes from popular items.
- **UserKNN gives the best balance of accuracy and diversity.** It is nearly as accurate as PureSVD (0.031), covers more than 8k items and has a Gini of 0.25.
- **MostPop is a strong baseline.** MF, PMF, NonNegMF and AttributeItemKNN all score below it. BPRMF scores about the same as MostPop and only covers 25 items, so it is effectively recommending popular items.
- **The learned MF models look undertrained** (10 epochs, learning rate 0.001). PMF was still improving at its last epoch, and NonNegMF peaked at epoch 2.
- **Cosine similarity beats euclidean for ItemKNN** by about 9× in nDCG. The number of neighbors (50, 70 or 100) makes little difference.
- **AttributeItemKNN** has the highest coverage and Gini of any model except Random, but its accuracy is close to Random's.
- **All absolute scores are low,** which is expected with a sparse dataset and a temporal split.
